In [1]:
import re
import requests
import time

import pandas as pd
import pickle as pkl

In [ ]:
def extract_unique_terms(series, delimiters=r'[/;,]'):
    """Get unique terms from ClinVar dataframe columns"""
    all_terms = set()
    for val in series.dropna().unique():
        parts = re.split(delimiters, str(val))
        for p in parts:
            p = p.strip()
            if p:
                all_terms.add(p)
    return sorted(all_terms)

In [3]:
def validate_term(term, search_field):
    """Check if a term is recognized by ClinVar web search"""
    url = f'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=clinvar&term="{term}"[{search_field}]&retmode=json'
    resp = requests.get(url)
    data = resp.json()
    result = data.get("esearchresult", {})

    print(data)

    translation = result.get("querytranslation", "")
    error_list = result.get("errorlist", {})
    warning_list = result.get("warninglist", {})
    output_messages = warning_list.get("outputmessages", [])
    phrases_not_found = error_list.get("phrasesnotfound", [])
    no_items = any("no items found" in msg.lower() for msg in output_messages)
    
    field_demoted = "[all fields]" in translation.lower()
    syntax_error = any("syntax error" in msg.lower() for msg in output_messages)
    phrase_missing = len(phrases_not_found) > 0
    
    is_valid = not field_demoted and not syntax_error and not phrase_missing and not no_items

    time.sleep(0.5)

    return is_valid

In [4]:
# check Biomni's clinvar schema

clinvar_schema_path = 'biomni/tool/schema_db/clinvar.pkl'

with open(clinvar_schema_path, 'rb') as file:
    data = pkl.load(file)

print(data)

{
    "search_fields": {
        "gene": "[gene]",
        "gene_id": "[geneid]",
        "gene_full_name": "[gene_full_name]",
        "variant": "[varnam]",
        "disease": "[dis]",
        "clinical_significance": "[clinsig]",
        "variation_id": "[uid]",
        "rs_id": "[varacc]",
        "hgvs": "[varnam]",
        "chromosome": "[chr]",
        "coordinate": "[chrpos]",
        "GRCh37_coordinate": "[chrpos37]",
        "GRCh38_coordinate": "[chrpos]",
        "allele_id": "[alleleid]",
        "phenotype": "[dis]",
        "property": "[prop]",
        "molecular_consequence": "[molcons]",
        "review_status": "[revstat]",
        "type_of_variation": "[vartype]",
        "origin": "[origin]",
        "pubmed_id": "[pmid]",
        "trait_identifier": "[traitid]",
        "clinvar_accession": "[clv_acc]",
        "common_name": "[commonname]",
        "canonical_spdi": "[cspdi]",
        "creation_date": "[cdat]",
        "modification_date": "[mdat]",
        "cyto

In [6]:
# download ClinVar df

! curl ftp://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/variant_summary.txt.gz > database_validation/clinvar/variant_summary.txt.gz
! gunzip database_validation/clinvar/variant_summary.txt.gz

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  415M  100  415M    0     0  39.8M      0  0:00:10  0:00:10 --:--:-- 41.3M 0  38.8M      0  0:00:10  0:00:06  0:00:04 40.3M     0  39.5M      0  0:00:10  0:00:09  0:00:01 40.9M


In [8]:
# check ClinVar columns

clinvar_summary = pd.read_csv(f'database_validation/clinvar/variant_summary.txt', sep='\t')
clinvar_summary.columns

/var/folders/sf/_p6bwhy54t39rt1jt8rkkfhj5s_rsg/T/ipykernel_68912/327430341.py:3: DtypeWarning: Columns (18) have mixed types. Specify dtype option on import or set low_memory=False.
  clinvar_summary = pd.read_csv(f'database_validation/clinvar/variant_summary.txt', sep='\t')


Index(['#AlleleID', 'Type', 'Name', 'GeneID', 'GeneSymbol', 'HGNC_ID',
       'ClinicalSignificance', 'ClinSigSimple', 'LastEvaluated', 'RS# (dbSNP)',
       'nsv/esv (dbVar)', 'RCVaccession', 'PhenotypeIDS', 'PhenotypeList',
       'Origin', 'OriginSimple', 'Assembly', 'ChromosomeAccession',
       'Chromosome', 'Start', 'Stop', 'ReferenceAllele', 'AlternateAllele',
       'Cytogenetic', 'ReviewStatus', 'NumberSubmitters', 'Guidelines',
       'TestedInGTR', 'OtherIDs', 'SubmitterCategories', 'VariationID',
       'PositionVCF', 'ReferenceAlleleVCF', 'AlternateAlleleVCF',
       'SomaticClinicalImpact', 'SomaticClinicalImpactLastEvaluated',
       'ReviewStatusClinicalImpact', 'Oncogenicity',
       'OncogenicityLastEvaluated', 'ReviewStatusOncogenicity',
       'SCVsForAggregateGermlineClassification',
       'SCVsForAggregateSomaticClinicalImpact',
       'SCVsForAggregateOncogenicityClassification'],
      dtype='object')

In [9]:
# get valid vartypes

vartype_values = extract_unique_terms(clinvar_summary['Type'])
print(vartype_values)

valid_vartype = [vt.lower() for vt in vartype_values if validate_term(vt.lower(), 'vartype')]
print(valid_vartype)

['Complex', 'Deletion', 'Duplication', 'Indel', 'Insertion', 'Inversion', 'Microsatellite', 'Tandem duplication', 'Translocation', 'Variation', 'copy number gain', 'copy number loss', 'fusion', 'protein only', 'single nucleotide variant']
{'header': {'type': 'esearch', 'version': '0.3'}, 'esearchresult': {'count': '96', 'retmax': '20', 'retstart': '0', 'idlist': ['4813067', '4684994', '4532816', '4526699', '4526698', '4077123', '4056468', '3767251', '3068499', '3068439', '1810757', '1713199', '1712295', '1711134', '1703689', '1703665', '1703602', '1703582', '1703575', '1703560'], 'translationset': [], 'translationstack': [{'term': '"complex"[vartype]', 'field': 'vartype', 'count': '96', 'explode': 'N'}, 'GROUP'], 'querytranslation': '"complex"[vartype]'}}
{'header': {'type': 'esearch', 'version': '0.3'}, 'esearchresult': {'count': '179741', 'retmax': '20', 'retstart': '0', 'idlist': ['4813645', '4813644', '4813639', '4813637', '4813630', '4813614', '4813610', '4813605', '4813587', '481

In [ ]:
# get valid origins

origin_values = extract_unique_terms(clinvar_summary['Origin'])
print(origin_values)

valid_origins = [o for o in origin_values if validate_term(o, 'origin')]
print(valid_origins)

['biparental', 'de novo', 'germline', 'inherited', 'maternal', 'not applicable', 'not provided', 'not-reported', 'paternal', 'somatic', 'uniparental', 'unknown']
{'header': {'type': 'esearch', 'version': '0.3'}, 'esearchresult': {'count': '1851', 'retmax': '20', 'retstart': '0', 'idlist': ['4813489', '4813482', '4813458', '4813453', '4813082', '4813081', '4796686', '4796480', '4796015', '4795861', '4795860', '4795859', '4795858', '4795855', '4795250', '4759340', '4688169', '4688041', '4688040', '4688033'], 'translationset': [], 'translationstack': [{'term': '"biparental"[origin]', 'field': 'origin', 'count': '1851', 'explode': 'N'}, 'GROUP'], 'querytranslation': '"biparental"[origin]'}}
{'header': {'type': 'esearch', 'version': '0.3'}, 'esearchresult': {'count': '18384', 'retmax': '20', 'retstart': '0', 'idlist': ['4813647', '4813643', '4813642', '4813641', '4813637', '4813636', '4813626', '4813496', '4813495', '4813493', '4813486', '4813485', '4813480', '4813475', '4813474', '4813468'

In [11]:
# get valid revstats

status_values = sorted(clinvar_summary['ReviewStatus'].dropna().str.strip().str.lower().unique())
print(status_values)

valid_status = [s for s in status_values if validate_term(s, 'revstat')]
print(valid_status)

['-', 'criteria provided, conflicting classifications', 'criteria provided, multiple submitters, no conflicts', 'criteria provided, single submitter', 'no assertion criteria provided', 'no classification for the single variant', 'no classification provided', 'no classifications from unflagged records', 'practice guideline', 'reviewed by expert panel']
{'header': {'type': 'esearch', 'version': '0.3'}, 'esearchresult': {'count': '0', 'retmax': '0', 'retstart': '0', 'idlist': [], 'translationset': [], 'querytranslation': '', 'warninglist': {'phrasesignored': [], 'quotedphrasesnotfound': [], 'outputmessages': ['empty query', 'Query syntax error.']}}}
{'header': {'type': 'esearch', 'version': '0.3'}, 'esearchresult': {'count': '161066', 'retmax': '20', 'retstart': '0', 'idlist': ['4792855', '4773212', '4759175', '4759150', '4758852', '4758533', '4758185', '4757754', '4757507', '4756235', '4755995', '4746902', '4741058', '4739911', '4739672', '4739406', '4736453', '4729426', '4723506', '4711

In [12]:
# guess valid molcons

molcons_values = [
    "missense variant", "nonsense", "frameshift variant",
    "splice donor variant", "splice acceptor variant",
    "inframe deletion", "inframe insertion",
    "synonymous variant", "intron variant",
    "5 prime utr variant", "3 prime utr variant",
    "stop gained", "stop lost", "start lost",
    "no sequence alteration", "near-gene",
]

valid_molcons = [mc for mc in molcons_values if validate_term(mc, 'molcons')]
print(valid_molcons)

{'header': {'type': 'esearch', 'version': '0.3'}, 'esearchresult': {'count': '2496504', 'retmax': '20', 'retstart': '0', 'idlist': ['4813647', '4813643', '4813642', '4813638', '4813636', '4813635', '4813634', '4813633', '4813632', '4813631', '4813628', '4813625', '4813623', '4813622', '4813609', '4813608', '4813607', '4813606', '4813604', '4813603'], 'translationset': [], 'translationstack': [{'term': '"missense variant"[molcons]', 'field': 'molcons', 'count': '2496504', 'explode': 'N'}, 'GROUP'], 'querytranslation': '"missense variant"[molcons]'}}
{'header': {'type': 'esearch', 'version': '0.3'}, 'esearchresult': {'count': '102505', 'retmax': '20', 'retstart': '0', 'idlist': ['4813624', '4813605', '4813602', '4813586', '4813576', '4813573', '4813569', '4813568', '4813563', '4813546', '4813537', '4813534', '4813517', '4813489', '4813485', '4813482', '4813465', '4813462', '4813461', '4813459'], 'translationset': [], 'translationstack': [{'term': '"nonsense"[molcons]', 'field': 'molcons'

In [13]:
# More search fields for schema: https://www.ncbi.nlm.nih.gov/clinvar/docs/help/
# Property fields for schema: https://www.ncbi.nlm.nih.gov/clinvar/docs/properties/

In [ ]:
# API search link example: https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi?db=clinvar&term=PRKN[gene]+AND+"clinsig+pathogenic"[prop]+AND+("copy+number+gain"[vartype]+OR+"copy+number+loss"[vartype])&retmode=json&sort=relevance&retmax=20
# Web browser port example: https://www.ncbi.nlm.nih.gov/clinvar/?term=PRKN[gene]+AND+"clinsig+pathogenic"[prop]+AND+("copy+number+gain"[vartype]+OR+"copy+number+loss"[vartype])&sort=relevance